# Inventory Validator API Frontend

This notebook is a lightweight frontend for the FastAPI app.

It assumes:
- the API is already running locally;
- you will point `CSV_PATH` to a real CSV file;
- `httpx`, `pandas`, and Jupyter are available in your environment.


In [ ]:
from pathlib import Path
import json
import time

import httpx
import pandas as pd
from IPython.display import Markdown, display

BASE_URL = "http://127.0.0.1:8000"
TENANT_ID = "default"
CSV_PATH = Path("../path/to/your_inventory.csv")
POLL_INTERVAL_SECONDS = 1.0
DOWNLOAD_DIR = Path("../notebook_downloads")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Update CSV_PATH before running the notebook. Missing file: {CSV_PATH}"
    )

client = httpx.Client(base_url=BASE_URL, timeout=60.0)
display(Markdown(f"**Base URL:** `{BASE_URL}`  \n**Tenant:** `{TENANT_ID}`  \n**CSV:** `{CSV_PATH}`"))


In [ ]:
health_response = client.get("/health")
health_response.raise_for_status()
display(Markdown(f"API health: `{health_response.json()['status']}`"))

preview_df = pd.read_csv(CSV_PATH, dtype=str).fillna("")
display(preview_df.head(10))


In [ ]:
with CSV_PATH.open("rb") as csv_file:
    upload_response = client.post(
        "/validate",
        params={"tenant_id": TENANT_ID},
        files={"file": (CSV_PATH.name, csv_file, "text/csv")},
    )

upload_response.raise_for_status()
job_payload = upload_response.json()
JOB_ID = job_payload["job_id"]
job_payload


In [ ]:
def wait_for_job(job_id: str, poll_interval: float = POLL_INTERVAL_SECONDS) -> dict:
    while True:
        status_response = client.get(f"/jobs/{job_id}")
        status_response.raise_for_status()
        payload = status_response.json()
        status = payload["status"]
        print(
            f"job={job_id} status={status} total_rows={payload['total_rows']} issues={payload['total_issues']}"
        )
        if status in {"completed", "failed"}:
            return payload
        time.sleep(poll_interval)

job_status = wait_for_job(JOB_ID)
job_status


In [ ]:
if job_status["status"] != "completed":
    raise RuntimeError(f"Validation failed: {job_status.get('error_message')}")

result_response = client.get(f"/jobs/{JOB_ID}/result")
result_response.raise_for_status()
report_data = result_response.json()

summary_df = pd.DataFrame([report_data["summary"]])
display(Markdown("## Summary"))
display(summary_df)

row_results_df = pd.DataFrame(report_data["row_results"])
display(Markdown("## Row Results"))
display(row_results_df)

duplicates_df = pd.DataFrame(report_data["duplicates"])
display(Markdown("## Duplicates"))
display(duplicates_df if not duplicates_df.empty else pd.DataFrame([{"status": "No duplicates"}]))

grouped_rows = []
for code, occurrences in report_data["grouped_problems"].items():
    for occurrence in occurrences:
        grouped_rows.append({"code": code, **occurrence})

grouped_df = pd.DataFrame(grouped_rows)
display(Markdown("## Grouped Problems"))
display(grouped_df if not grouped_df.empty else pd.DataFrame([{"status": "No grouped problems"}]))


In [ ]:
json_output_path = DOWNLOAD_DIR / f"{JOB_ID}_result.json"
json_output_path.write_text(
    json.dumps(report_data, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

pdf_response = client.get(f"/jobs/{JOB_ID}/report")
pdf_response.raise_for_status()
pdf_output_path = DOWNLOAD_DIR / f"{JOB_ID}_report.pdf"
pdf_output_path.write_bytes(pdf_response.content)

display(Markdown("## Downloaded Artifacts"))
display(
    {
        "json_result": str(json_output_path.resolve()),
        "pdf_report": str(pdf_output_path.resolve()),
    }
)


In [ ]:
client.close()
